# Data Acquisition — DEM, Sentinel-2, Landsat, OSM Streets, Overture Maps

**Library:** [`sitex`](../sitex) · **Original, fully-documented notebooks:**
[`00-Data-Acquisition_RemoteSensing.ipynb`](../documentations/00-Data-Acquisition_RemoteSensing.ipynb) ·
[`00-Data-OSM_Streetnetwork.ipynb`](../documentations/00-Data-OSM_Streetnetwork.ipynb) ·
[`00-Data-Overture_Extractor.ipynb`](../documentations/00-Data-Overture_Extractor.ipynb)

One combined data-acquisition notebook instead of three. All the fetch logic lives in
`sitex.data`; this notebook is the AOI, the parameters, and the calls.

**Two ways to set the area of interest (AOI), shared by every source below:**
- **Method 1 — place name (default).** Geocodes an OSM administrative boundary for
  `PLACE` and uses its bounding box + polygon. Validate the name on
  [openstreetmap.org](https://www.openstreetmap.org/) first.
- **Method 2 — interactive map.** Draw a rectangle on a Leafmap widget, or keep the
  default 2 km box around `local_lat`/`local_lon`.

**Outputs written to disk (`data/`):**

| File | Section |
|---|---|
| `dem/dtm_nasadem.tif`, `dem/dsm_aw3d30.tif` | 4 — needs your own OpenTopography key |
| `sentinel2/s2_{year}_B04_B08_B11.tif` | 5 |
| `landsat/output/{LANDSAT_ID}/..._clipped.tif`, `scene_info.json` | 6 |
| `osm/{slug}_{network_type}.gpkg` | 7 |
| `overture/{slug}_overture.gpkg` + per-theme GeoPackages | 8 |

**`scene_info.json` is the handoff file** for downstream analysis notebooks — see
Section 6.


## 1. Install & import

In [ ]:
# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[data] @ git+https://github.com/ArchiColab/sitex.git"

import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive")
    %pip install "sitex[data] @ git+https://github.com/ArchiColab/sitex.git"


Running in: Local environment


In [2]:
from pathlib import Path

from sitex.data import aoi as aoi_lib
from sitex.data import remote_sensing as rs
from sitex.data import street_network as sn
from sitex.data import overture as ovt

print("Imports OK.")


Imports OK.


## 2. Output directories

In [3]:
DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs") if IN_COLAB else Path("..") / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEM_DIR = DATA_DIR / "dem"
SEN_DIR = DATA_DIR / "sentinel2"
LS_DIR = DATA_DIR / "landsat" / "output"
OSM_DIR = DATA_DIR / "osm"
OVT_DIR = DATA_DIR / "overture"
for d in (DEM_DIR, SEN_DIR, LS_DIR, OSM_DIR, OVT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"DEM        : {DEM_DIR.resolve()}")
print(f"Sentinel-2 : {SEN_DIR.resolve()}")
print(f"Landsat    : {LS_DIR.resolve()}")
print(f"OSM streets: {OSM_DIR.resolve()}")
print(f"Overture   : {OVT_DIR.resolve()}")


DEM        : C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\dem
Sentinel-2 : C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\sentinel2
Landsat    : C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\landsat\output
OSM streets: C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\osm
Overture   : C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\overture


## 3. Area of Interest (defined once)

Sections 4-8 all reuse the `aoi` object produced here, so every source agrees on one
bounding box regardless of which method resolved it.

In [4]:
# Select AOI method: 1 = place name (default)  |  2 = interactive map
METHOD = 1

# Method 1 — place name (validate on https://www.openstreetmap.org/ before running)
PLACE = "Phường Pleiku, Gia Lai, Vietnam"

# Method 2 / map centering — Pleiku city centre
local_lat = 13.9833
local_lon = 108.0000

# Landsat search window (Section 6)
BEGIN_DATE = "2025-11-01"   # Dry season start (lower cloud cover)
END_DATE   = "2026-04-30"   # Dry season end
MAX_CLOUD  = 20             # Maximum cloud cover (%)


In [5]:
#@title <font color=#1B7192> Method 1 — AOI from place name </font>  { display-mode: "form" }
if METHOD == 1:
    print(f"Fetching OSM boundary for '{PLACE}'...")
    aoi = aoi_lib.resolve_aoi_by_place(PLACE)
    local_lat, local_lon = aoi.center_lat, aoi.center_lon
    west, south, east, north = aoi.bbox
    print(f"BBox: (xmin={west:.5f}, ymin={south:.5f}, xmax={east:.5f}, ymax={north:.5f})")
    print(f"Centroid: ({aoi.center_lat:.5f}, {aoi.center_lon:.5f})")


Fetching OSM boundary for 'Phường Pleiku, Gia Lai, Vietnam'...
BBox: (xmin=107.99506, ymin=13.96561, xmax=108.06338, ymax=14.03363)
Centroid: (13.99661, 108.03049)


In [6]:
#@title <font color=#1B7192> Method 2 — AOI from interactive map </font>  { display-mode: "form" }
if METHOD == 2:
    aoi_map, _default_bbox = aoi_lib.build_aoi_map(local_lat, local_lon, dist_m=1000, zoom=14)
    print("Draw a rectangle to customise the AOI. Yellow = default 2 km box. "
          "Run the next cell to confirm.")
    aoi_map


In [7]:
#@title <font color=#1B7192> Confirm AOI </font>  { display-mode: "form" }
if METHOD == 2:
    aoi = aoi_lib.resolve_aoi_from_map(aoi_map, _default_bbox, slug="custom_aoi")
    local_lat, local_lon = aoi.center_lat, aoi.center_lon

print("\nFinal AOI (used for DEM, Sentinel-2, Landsat, streets, and Overture below):")
west, south, east, north = aoi.bbox
print(f"  SW: ({south:.6f}, {west:.6f})  NE: ({north:.6f}, {east:.6f})")
print(f"  Auto-derived LOCAL_EPSG: {aoi.local_epsg}")



Final AOI (used for DEM, Sentinel-2, Landsat, streets, and Overture below):
  SW: (13.965607, 107.995063)  NE: (14.033631, 108.063382)
  Auto-derived LOCAL_EPSG: 32649


## 4. DEM / DSM download

- **DTM** (NASADEM) — bare-earth elevation (radar penetrates canopy)
- **DSM** (AW3D30) — top surface, includes buildings/trees

**Needs your own free OpenTopography API key** ([get one here](https://portal.opentopography.org/)) —
this cell is skipped by default so it doesn't fail (or falsely claim success) when run
without one. Fill in `YOUR_OPENTOPOGRAPHY_API_KEY` below and flip `RUN_DEM = True`
yourself once you have a key.

In [ ]:
YOUR_OPENTOPOGRAPHY_API_KEY = "YOUR API KEY"    #@param {type:"string"}
#@markdown <font color=#1B7192> Get one free at: https://portal.opentopography.org/ </font>

RUN_DEM = False  # flip to True once you've filled in a real key above

if RUN_DEM:
    dtm_path, dsm_path = rs.download_dem(aoi, DEM_DIR, api_key=YOUR_OPENTOPOGRAPHY_API_KEY)
    print(f"DTM exported: {dtm_path}")
    print(f"DSM exported: {dsm_path}")
else:
    dtm_path = dsm_path = None
    print("DEM download skipped (RUN_DEM = False). Set your OpenTopography API key "
          "above, then flip RUN_DEM = True and re-run this cell.")


DTM exported: ..\..\data\dem\dtm_nasadem.tif
DSM exported: ..\..\data\dem\dsm_aw3d30.tif


## 5. Sentinel-2 annual composites (OpenEO / CDSE)

Server-side median composite per year — **B04** (Red), **B08** (NIR), **B11** (SWIR,
needed for NDBI in the Urban Change Detection notebook). Backend:
[Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/) (free, one-time
account registration — first run opens a browser tab for login).

In [10]:
#@title <font color=#1B7192> Connect to Copernicus Data Space Ecosystem (CDSE) </font>  { display-mode: "form" }
connection = rs.connect_cdse()
print("Connected to CDSE.")


Authenticated using refresh token.
Connected to CDSE.


In [11]:
#@title <font color=#1B7192> Download Sentinel-2 annual composites </font>  { display-mode: "form" }
YEAR_EARLY  = 2017    #@param {type:"number"}
YEAR_RECENT = 2026    #@param {type:"number"}

s2_paths = {}
for year in [YEAR_EARLY, YEAR_RECENT]:
    path = SEN_DIR / f"s2_{year}_B04_B08_B11.tif"
    s2_paths[year] = rs.download_sentinel2_annual_composite(connection, aoi, year, path)
    with __import__("rasterio").open(s2_paths[year]) as src:
        print(f"{year} -> {s2_paths[year]}  (CRS: {src.crs})")


2017 -> ..\..\data\sentinel2\s2_2017_B04_B08_B11.tif  (CRS: EPSG:32649)
2026 -> ..\..\data\sentinel2\s2_2026_B04_B08_B11.tif  (CRS: EPSG:32649)


## 6. Landsat 8/9 bands 4, 5, 10 (STAC, Level-1 → Level-2 fallback)

Queries the public Landsat Collection 2 STAC catalog on Microsoft Planetary Computer —
no credentials required.

**Level-1 vs Level-2:** Level-1 (raw digital numbers) is what a manual
NDVI/Emissivity/LST calculation is normally written for. USGS hasn't republished a
Level-1 product for every scene in Collection 2 though — coverage is patchier for
scenes downlinked through international ground stations, common for Southeast Asia.
Level-2 (atmospherically corrected Surface Reflectance + a ready-made Surface
Temperature band) has much more complete coverage. So: **try Level-1 first, fall back
to Level-2 automatically**. The UHI notebook reads `PRODUCT_LEVEL` from
`scene_info.json` and adapts its formulas accordingly.

In [12]:
items, PRODUCT_LEVEL, COLLECTION = rs.search_landsat(aoi, BEGIN_DATE, END_DATE, MAX_CLOUD)
if PRODUCT_LEVEL == "L2":
    print("No scenes found in 'landsat-c2-l1' for this AOI/date range - falling back to "
          "Level-2. This is a normal Landsat archive gap, not an error.\n")

print(f"Using collection: {COLLECTION}  (PRODUCT_LEVEL = {PRODUCT_LEVEL})")
print(f"Found {len(items)} scenes.\n")
for i, it in enumerate(items):
    props = it.properties
    cloud = props.get("eo:cloud_cover")
    cloud_str = f"{cloud:.1f}%" if cloud is not None else "?"
    print(f"[{i:02d}] {it.id}  cloud={cloud_str}  date={props.get('datetime', '?')[:10]}  "
          f"platform={props.get('platform', '?')}")


No scenes found in 'landsat-c2-l1' for this AOI/date range - falling back to Level-2. This is a normal Landsat archive gap, not an error.

Using collection: landsat-c2-l2  (PRODUCT_LEVEL = L2)
Found 4 scenes.

[00] LC09_L2SP_124050_20260429_02_T1  cloud=12.7%  date=2026-04-29  platform=landsat-9
[01] LC08_L2SP_124050_20260405_02_T1  cloud=3.0%  date=2026-04-05  platform=landsat-8
[02] LC09_L2SP_124050_20260328_02_T1  cloud=0.1%  date=2026-03-28  platform=landsat-9
[03] LC08_L2SP_124050_20260320_02_T1  cloud=12.0%  date=2026-03-20  platform=landsat-8


In [13]:
# --- Pick the scene index to process ------------------------------------------
SCENE_INDEX = 0    # <-- change to pick a different date/scene

item = items[SCENE_INDEX]
cloud = item.properties.get("eo:cloud_cover")
print(f"Selected: {item.id}")
print(f"Cloud cover: {cloud:.1f}%" if cloud is not None else "Cloud cover: ?")
print(f"Acquisition date: {item.properties.get('datetime', '?')}")


Selected: LC09_L2SP_124050_20260429_02_T1
Cloud cover: 12.7%
Acquisition date: 2026-04-29T03:06:34.671474Z


In [14]:
#@title <font color=#1B7192> Download & clip bands 4, 5, 10 </font>  { display-mode: "form" }
band_paths = rs.download_landsat_bands(item, aoi, LS_DIR / item.id)
for band, path in band_paths.items():
    print(f"Saved {path}")

with __import__("rasterio").open(band_paths["B4"]) as src:
    print(f"\nClipped shape: {src.read(1).shape}   CRS: {src.crs}   Resolution (m): {src.res}")


Saved ..\..\data\landsat\output\LC09_L2SP_124050_20260429_02_T1\LC09_L2SP_124050_20260429_02_T1_B4_clipped.tif
Saved ..\..\data\landsat\output\LC09_L2SP_124050_20260429_02_T1\LC09_L2SP_124050_20260429_02_T1_B5_clipped.tif
Saved ..\..\data\landsat\output\LC09_L2SP_124050_20260429_02_T1\LC09_L2SP_124050_20260429_02_T1_B10_clipped.tif

Clipped shape: (255, 250)   CRS: EPSG:32649   Resolution (m): (30.0, 30.0)


## 6b. Save unified `scene_info.json`

Combines the AOI, DEM paths, Sentinel-2 annual composite paths, and Landsat fields
into one metadata file — the handoff downstream analysis notebooks read from.

In [15]:
scene_info_path = LS_DIR / item.id / "scene_info.json"
rs.save_scene_info(
    aoi, scene_info_path,
    dem_paths=(dtm_path, dsm_path) if dtm_path else None,
    sentinel2_paths=s2_paths,
    landsat_item=item,
    product_level=PRODUCT_LEVEL,
    band_paths=band_paths,
)
print(f"Scene metadata saved to: {scene_info_path}")
print(f"Product level: {PRODUCT_LEVEL}")


Scene metadata saved to: ..\..\data\landsat\output\LC09_L2SP_124050_20260429_02_T1\scene_info.json
Product level: L2


## 7. OSM Street Network

Downloads drive / walk / drive_service networks, truncated to the AOI boundary (the
place's real administrative shape for Method 1, or the drawn rectangle for Method 2)
and reprojected to the local UTM zone.

In [16]:
#@title <font color=#1B7192> Download street networks </font>  { display-mode: "form" }
NETWORK_TYPES = ("drive", "walk", "drive_service")

graphs = sn.download_street_networks(aoi, NETWORK_TYPES)
street_paths = sn.save_street_networks(graphs, OSM_DIR, aoi.slug)
for network_type, path in street_paths.items():
    n_edges = graphs[network_type].number_of_edges()
    print(f"{network_type:14s} {n_edges:>6,} edges  -> {path}")


drive           4,146 edges  -> ..\..\data\osm\phường_pleiku_drive.gpkg
walk            6,688 edges  -> ..\..\data\osm\phường_pleiku_walk.gpkg
drive_service   6,122 edges  -> ..\..\data\osm\phường_pleiku_drive_service.gpkg


## 8. Overture Maps (buildings, roads, places, land use, water)

DuckDB range-requests against Overture's S3 GeoParquet files — no full dataset
download needed. Data is clipped to the same AOI boundary as the street network
above.

In [17]:
#@title <font color=#1B7192> Check release access </font>  { display-mode: "form" }
LATEST_RELEASE = ovt.get_latest_release()
print(f"Using Overture release: {LATEST_RELEASE}")
ovt.check_release(LATEST_RELEASE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Using Overture release: 2026-08-19.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Release '2026-08-19.0' is accessible. Row count sample: 2529582613


True

In [18]:
#@title <font color=#1B7192> Extract Overture layers </font>  { display-mode: "form" }
LAYERS_TO_EXTRACT = ["buildings", "roads", "places", "land_use", "water"]

overture_result = ovt.extract_by_aoi(aoi, OVT_DIR, LAYERS_TO_EXTRACT, LATEST_RELEASE)
print(f"\nActive output: {overture_result['main']}")



Extracting 'buildings'...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 21,287 features

Extracting 'roads'...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 2,886 features

Extracting 'places'...


...

  -> 3,380 features

Extracting 'land_use'...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 27 features

Extracting 'water'...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 4 features

Main       -> C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\overture\phường_pleiku_overture.gpkg
Streets    -> C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\overture\phường_pleiku_streets.gpkg
Landuse    -> C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\overture\phường_pleiku_landuse.gpkg
Buildings  -> C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\overture\phường_pleiku_buildings.gpkg
Places     -> C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\overture\phường_pleiku_places.gpkg
Layers written: ['BLDG', 'ROADS', 'POIS', 'LULC', 'WATER']

Active output: ..\..\data\overture\phường_pleiku_overture.gpkg


## 9. Summary

In [19]:
print(f"{'='*55}")
print("  Data Acquisition Summary")
print(f"{'='*55}")
print(f"  AOI                : {aoi.slug}  (LOCAL_EPSG={aoi.local_epsg})")
print(f"  DEM                : {'downloaded' if dtm_path else 'SKIPPED (needs your API key, Section 4)'}")
print(f"  Sentinel-2 years   : {list(s2_paths.keys())}")
print(f"  Landsat scene      : {item.id}  ({PRODUCT_LEVEL})")
print(f"  Street networks    : {list(street_paths.keys())}")
print(f"  Overture layers    : {overture_result['written']}")
print(f"\n  scene_info.json    : {scene_info_path}")
print("\nNext: open 02-ENV-UHI_Analysis.ipynb, 02-ENV-Urban_Change_Detection.ipynb, "
      "or the Space Syntax / Accessibility notebooks — they read the files written above.")


  Data Acquisition Summary
  AOI                : phường_pleiku  (LOCAL_EPSG=32649)
  DEM                : downloaded
  Sentinel-2 years   : [2017, 2026]
  Landsat scene      : LC09_L2SP_124050_20260429_02_T1  (L2)
  Street networks    : ['drive', 'walk', 'drive_service']
  Overture layers    : ['BLDG', 'ROADS', 'POIS', 'LULC', 'WATER']

  scene_info.json    : ..\..\data\landsat\output\LC09_L2SP_124050_20260429_02_T1\scene_info.json

Next: open 02-ENV-UHI_Analysis.ipynb, 02-ENV-Urban_Change_Detection.ipynb, or the Space Syntax / Accessibility notebooks — they read the files written above.
